# 입낚 물고기 감지 모델 학습
**YOLOv8n-pose → TensorFlow.js 변환**

**순서:**
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 시작
2. 각 셀을 순서대로 실행 (▶ 버튼 또는 Shift+Enter)
3. 셀 3에서 Roboflow API 키 입력
4. 학습 완료 후 zip 파일 자동 다운로드됨

In [ ]:
# ─── 셀 1: GPU 확인 ───────────────────────────────────────────
import torch
gpu = torch.cuda.is_available()
print(f"GPU 사용 가능: {gpu}")
if gpu:
    print(f"GPU 모델: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 재시작하세요.")

In [ ]:
# ─── 셀 2: 패키지 설치 ────────────────────────────────────────
!pip install ultralytics roboflow --quiet
print("✅ 설치 완료")

In [ ]:
# ─── 셀 3: Roboflow 데이터셋 다운로드 ─────────────────────────
# Roboflow > 프로젝트 > Versions > Export > Show download code 에서 복사
from roboflow import Roboflow

ROBOFLOW_API_KEY = "여기에_API_키_입력"   # ← 수정 필요
WORKSPACE        = "여기에_워크스페이스"  # ← 수정 필요  
PROJECT          = "여기에_프로젝트명"    # ← 수정 필요
VERSION          = 1                      # ← 버전 번호

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")
print(f"✅ 데이터셋 다운로드 완료: {dataset.location}")

In [ ]:
# ─── 셀 4: 학습 ───────────────────────────────────────────────
# 사진 수가 적으면 epochs=50, 많으면 100~150
from ultralytics import YOLO

model = YOLO("yolov8n-pose.pt")  # nano (가벼움, 모바일 최적)

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="fish_detector",
    patience=20,        # 20 에폭 개선 없으면 조기 종료
    device=0 if torch.cuda.is_available() else "cpu",
    plots=True,         # 학습 곡선 시각화
)

print("\n✅ 학습 완료")
print(f"최적 모델: runs/pose/fish_detector/weights/best.pt")

In [ ]:
# ─── 셀 5: 검증 (정확도 확인) ─────────────────────────────────
best = YOLO("runs/pose/fish_detector/weights/best.pt")
metrics = best.val()
print(f"\n📊 결과")
print(f"  Box mAP50:  {metrics.box.map50:.3f}")
try:
    print(f"  Pose mAP50: {metrics.pose.map50:.3f}")
except:
    pass
print("\n※ 0.5 이상이면 실용적, 0.7 이상이면 우수")

In [ ]:
# ─── 셀 6: TF.js 변환 ─────────────────────────────────────────
best = YOLO("runs/pose/fish_detector/weights/best.pt")
best.export(format="tfjs")
print("✅ TF.js 변환 완료")

import os
# 변환된 파일 위치 찾기
for root, dirs, files in os.walk("runs"):
    for f in files:
        if f == "model.json":
            print(f"  모델 위치: {root}")

In [ ]:
# ─── 셀 7: 다운로드 ───────────────────────────────────────────
import shutil, os, glob
from google.colab import files

# model.json 이 있는 폴더를 찾아서 zip 압축
model_dirs = glob.glob("runs/**/*_web_model", recursive=True)
if not model_dirs:
    model_dirs = glob.glob("runs/**/best_web_model", recursive=True)

if model_dirs:
    export_dir = model_dirs[0]
    zip_path = "/content/fish_detector_tfjs"
    shutil.make_archive(zip_path, "zip", export_dir)
    print(f"📦 압축 완료: {zip_path}.zip")
    files.download(f"{zip_path}.zip")
    print("\n⬇️  다운로드 시작됨")
    print("압축 해제 후 모든 파일을:")
    print("  입낚/public/models/fish_detector/ 폴더에 복사하세요")
else:
    print("⚠️  변환 파일을 찾지 못했습니다. 셀 6을 다시 실행하세요.")